<a href="https://colab.research.google.com/github/oit-cpai/cpai-foundations/blob/main/solutions/03_mdp_bellman_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# R3 MDP とベルマン方程式(notebook)

本文:[`03-mdp-bellman.md`](https://github.com/oit-cpai/cpai-foundations/blob/main/rl/03-mdp-bellman.md)

**進め方**:予想 → 手計算 → コードで確かめる → 実装(`TODO`)→ 条件を変える → 説明。`check(...)` が `OK` と出るまで直してから次に進む。上のセルから順に実行すること。

> **これは解答つきの版である。** まず演習の版(`rl/03_mdp_bellman.ipynb`)を自分で解き、答え合わせに使うこと。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def check(name, got, expected, tol=1e-8):
    got, expected = np.asarray(got, dtype=float), np.asarray(expected, dtype=float)
    if got.shape != expected.shape:
        print(f"NG  {name}: 形が違う  got {got.shape}, expected {expected.shape}")
    elif np.allclose(got, expected, atol=tol):
        print(f"OK  {name}")
    else:
        print(f"NG  {name}: got {got}, expected {expected}")

## 1. 収益

**予想**:報酬の列が −1、−1、−1、+10 のとき、割引率 $\gamma$ を 1 から 0 に小さくすると、収益 $G_0$ はどう変わるか。

- 予想:(ここに書く)

**手計算**:本文の確認課題1 を解く。

- 手計算:(ここに書く)

**実装**:報酬の列 `rewards`($r_1, r_2, \dots$)から、すべての時刻の収益 $G_0, G_1, \dots$ を返す関数 `returns` を書く。本文の式 (2) $G_t = r_{t+1} + \gamma G_{t+1}$ を使い、**最後の時刻から前へ**計算する(最後の報酬のあとの収益は 0)。

In [ ]:
def returns(rewards, gamma):
    G = np.zeros(len(rewards))
    future = 0.0
    for t in reversed(range(len(rewards))):
        future = rewards[t] + gamma * future
        G[t] = future
    return G

rewards = [-1, -1, -1, 10]
check("γ = 0   のとき G0 = -1", returns(rewards, 0.0)[0], -1.0)
check("γ = 0.5 のとき G0 = -0.5", returns(rewards, 0.5)[0], -0.5)
check("γ = 0.9 のとき G0 = 4.58", returns(rewards, 0.9)[0], 4.58)
check("γ = 1   のとき G0 = 7", returns(rewards, 1.0)[0], 7.0)
G = returns(rewards, 0.9)
check("式 (2):G0 = r1 + γ G1", G[0], rewards[0] + 0.9 * G[1])
print("γ = 0.9 のときの G0〜G3:", np.round(G, 3))

## 2. 小さな MDP を配列で書く

本文 §6 の MDP(状態 3 つ、行動 2 つ)を、遷移確率の配列 `P_small[s, a, s2]`(形 `(3, 2, 3)`)と報酬の配列 `R_small[s, a]`(形 `(3, 2)`)で書く。

**実装**:本文の表のとおりに値を入れる。

In [ ]:
nS, nA = 3, 2          # 行動 0 = 待つ、1 = 進む
P_small = np.zeros((nS, nA, nS))
R_small = np.zeros((nS, nA))
P_small[0, 0, 0] = 1.0;  R_small[0, 0] = 0.0
P_small[0, 1, 1] = 1.0;  R_small[0, 1] = -1.0
P_small[1, 0, 1] = 1.0;  R_small[1, 0] = 0.0
P_small[1, 1, 2] = 0.5;  P_small[1, 1, 1] = 0.5;  R_small[1, 1] = 2.0
P_small[2, :, 2] = 1.0   # 状態2 は吸収状態(報酬 0)

check("各 (s, a) で、遷移確率の和は 1", P_small.sum(axis=2), np.ones((nS, nA)))
check("状態1 で進む:確率 0.5 で状態2、0.5 で状態1 に残る", P_small[1, 1], [0, 0.5, 0.5])
check("報酬", R_small, [[0, -1], [0, 2], [0, 0]])

## 3. 方策の価値を、連立一次方程式で解く

**手計算**:本文 §6 の「いつも進む」の価値 $V(0) = 25/11$、$V(1) = 40/11$ を、自分で導く。

- 手計算:(ここに書く)

**実装**:決定的な方策(各状態で選ぶ行動の配列)の価値を、本文の式 (4) で求める関数 `policy_value` を書く(M4 §4 と同じ)。

In [ ]:
def policy_value(P, R, policy, gamma):
    # 決定的な方策 policy(各状態で選ぶ行動)の価値を、(I - gamma P_pi) v = r_pi で求める
    idx = np.arange(P.shape[0])
    P_pi, r_pi = P[idx, policy], R[idx, policy]
    return np.linalg.solve(np.eye(P.shape[0]) - gamma * P_pi, r_pi)

gamma = 0.9
always_go = np.array([1, 1, 1])
wait_at_0 = np.array([0, 1, 1])
check("いつも進む:V = (25/11, 40/11, 0)", policy_value(P_small, R_small, always_go, gamma), [25 / 11, 40 / 11, 0])
check("状態0 で待つ:V(0) = 0", policy_value(P_small, R_small, wait_at_0, gamma)[0], 0.0)

## 4. 確率的な方策と、ベルマン方程式

**実装**:確率的な方策 `pi[s, a]`(各行の和が 1)の価値を求める関数 `policy_value_stochastic` を書く。決定的な方策を「選ぶ行動の確率が 1」と書き直すと、同じ値になることも確かめる。

**確かめる**:求めた価値を、本文の式 (3) の右辺に入れると、同じ値に戻る(ベルマン方程式を満たす)。

In [ ]:
def policy_value_stochastic(P, R, pi, gamma):
    # 確率的な方策 pi[s, a] の価値
    P_pi = np.einsum("sa,sat->st", pi, P)      # P_pi[s, s2] = Σ_a pi(a|s) P(s2|s, a)
    r_pi = (pi * R).sum(axis=1)                # r_pi[s] = Σ_a pi(a|s) R(s, a)
    return np.linalg.solve(np.eye(P.shape[0]) - gamma * P_pi, r_pi)

one_hot = np.eye(nA)[always_go]                # 決定的な方策を、確率的な方策の形にしたもの
check("決定的な方策と同じ値になる", policy_value_stochastic(P_small, R_small, one_hot, gamma),
      policy_value(P_small, R_small, always_go, gamma))
uniform = np.full((nS, nA), 0.5)
V_uniform = policy_value_stochastic(P_small, R_small, uniform, gamma)
rhs = (uniform * (R_small + gamma * P_small @ V_uniform)).sum(axis=1)   # 式 (3) の右辺
check("求めた価値は、ベルマン方程式 (3) を満たす", V_uniform, rhs)
print("半々の方策の価値:", np.round(V_uniform, 3))

**価値は、収益の平均の極限である**(本文 §4)。半々の方策で、状態0 から実際にエピソードを何度も試し、収益の平均が $V(0)$ に近づくことを確かめる。

In [ ]:
def sample_return(P, R, pi, s, gamma, rng, max_steps=300):
    # 方策 pi で状態 s から動かし、割引した収益を1つ返す
    G, discount = 0.0, 1.0
    for _ in range(max_steps):
        a = rng.choice(P.shape[1], p=pi[s])
        G += discount * R[s, a]
        discount *= gamma
        s = rng.choice(P.shape[0], p=P[s, a])
    return G

rng = np.random.default_rng(0)
samples = np.array([sample_return(P_small, R_small, uniform, 0, gamma, rng) for _ in range(4000)])
print(f"収益の平均 {samples.mean():.3f}(標準誤差 {samples.std() / np.sqrt(len(samples)):.3f})   式で求めた V(0) = {V_uniform[0]:.3f}")

## 5. 行動の価値 $Q$

**手計算**:本文の確認課題2(方策「いつも進む」の $Q(0, \text{待つ})$ と $Q(0, \text{進む})$)を解く。

- 手計算:(ここに書く)

本文 §5 の式で $Q$ を計算し、 $V$ が「方策で $Q$ を平均したもの」になることを確かめる。

In [ ]:
V_go = policy_value(P_small, R_small, always_go, gamma)
Q_go = R_small + gamma * P_small @ V_go          # 形 (3, 2)
print("Q(s, a) =\n", np.round(Q_go, 4))
check("Q(0, 待つ) = 0.9 × 25/11", Q_go[0, 0], 0.9 * 25 / 11)
check("Q(0, 進む) = 25/11", Q_go[0, 1], 25 / 11)
check("V(s) = Q(s, 方策が選ぶ行動)", V_go, Q_go[np.arange(nS), always_go])

## 6. 最適な価値:決定的な方策をすべて試す

**予想**:本文 §6 の MDP で、最適な方策は何か。

- 予想:(ここに書く)

**実装**:決定的な方策は $2^3 = 8$ 通りしかない。すべての方策の価値を求め、状態ごとの最大値を `V_star` とする(本文 §7)。そのうえで、**ひとつの方策がすべての状態で同時に最大になっている**ことと、`V_star` がベルマン最適方程式 (5) を満たすことを確かめる。

In [ ]:
import itertools

all_values = []
for policy in itertools.product(range(nA), repeat=nS):          # 2^3 = 8 通り
    all_values.append(policy_value(P_small, R_small, np.array(policy), gamma))
all_values = np.array(all_values)                               # 形 (8, 3)
V_star = all_values.max(axis=0)                                 # 状態ごとの最大値

check("最適な価値 V* = (25/11, 40/11, 0)", V_star, [25 / 11, 40 / 11, 0])
best = [p for p, v in zip(itertools.product(range(nA), repeat=nS), all_values) if np.allclose(v, V_star)]
print("すべての状態で V* を与える方策:", best)
rhs_opt = (R_small + gamma * P_small @ V_star).max(axis=1)      # 式 (5) の右辺
check("V* は、ベルマン最適方程式 (5) を満たす", V_star, rhs_opt)

状態2 では行動がどちらでも同じなので、最適な方策は2つ見つかる(状態0 と状態1 では「進む」)。

R2 のグリッドでは、決定的な方策が $4^{16}$ 通りあるので、この方法は使えない。R4 で、くり返しの計算で解く。

## 7. R2 のグリッドを MDP として書く

R2 で `step()` として作ったグリッドを、遷移確率の配列 `P[s, a, s2]`(形 `(16, 4, 16)`)と報酬 `R[s, a]` で書く。`next_state` は R2 で作ったものを用意してある。

**手計算**:滑る床(`slip = 0.2`)で、マス0 で「右」を選んだときの次のマスの確率と、マス1 で「下」を選んだときの報酬の期待値を求める。

- 手計算:(ここに書く)

**実装**:`build_grid_mdp` を書く。

In [ ]:
# R2〜R5 で使うグリッド(本文 R2 §3)
N_ROW = N_COL = 4
S, A = N_ROW * N_COL, 4
START, GOAL, MUD = 0, 15, {5, 10}
MOVES = [(-1, 0), (0, 1), (1, 0), (0, -1)]   # 0=上、1=右、2=下、3=左
ARROWS = ["↑", "→", "↓", "←"]

def to_pos(s):
    return divmod(s, N_COL)

def next_state(s, a):
    # 行動 a の向きに1マス進む。外に出るなら、その場にとどまる(R2 §2)
    r, c = to_pos(s)
    dr, dc = MOVES[a]
    r2, c2 = r + dr, c + dc
    return r2 * N_COL + c2 if (0 <= r2 < N_ROW and 0 <= c2 < N_COL) else s

def build_grid_mdp(slip=0.2, mud_reward=-10.0):
    P = np.zeros((S, A, S))
    R = np.zeros((S, A))
    for s in range(S):
        for a in range(A):
            if s == GOAL:                       # ゴールは吸収状態:とどまり、報酬 0
                P[s, a, s] = 1.0
                continue
            for a2 in range(A):                 # 実際に進む方向 a2 の確率
                p = (1 - slip) * (a2 == a) + slip / A
                P[s, a, next_state(s, a2)] += p
            R[s, a] = sum(P[s, a, s2] * (mud_reward if s2 in MUD else -1.0) for s2 in range(S))
    return P, R

P_grid, R_grid = build_grid_mdp(slip=0.2)
check("遷移確率の和は 1", P_grid.sum(axis=2), np.ones((S, A)))
check("マス0 で右:マス1 へ 0.85、とどまる 0.10、マス4 へ 0.05", P_grid[0, 1, [1, 0, 4]], [0.85, 0.10, 0.05])
check("マス0 で右の報酬:-1", R_grid[0, 1], -1.0)
check("マス1 で下(ぬかるみへ):0.85 × (-10) + 0.15 × (-1) = -8.65", R_grid[1, 2], -8.65)
check("ゴールは吸収状態", [P_grid[GOAL, 2, GOAL], R_grid[GOAL, 2]], [1.0, 0.0])

R2 の方策「右→下」の価値を、式 (4) で求める。R2 ではエピソードを何度も試して平均を求めたが、ここでは1回の計算で、すべてのマスの価値が分かる。シミュレーションの平均とも比べる。

In [ ]:
gamma = 0.95
right_then_down = np.array([1 if to_pos(s)[1] < N_COL - 1 else 2 for s in range(S)])
V_rd = policy_value(P_grid, R_grid, right_then_down, gamma)
print("方策「右→下」の価値(γ = 0.95):")
print(np.round(V_rd.reshape(N_ROW, N_COL), 2))

pi_rd = np.eye(A)[right_then_down]
rng = np.random.default_rng(1)
samples = np.array([sample_return(P_grid, R_grid, pi_rd, START, gamma, rng) for _ in range(3000)])
print(f"シミュレーションの収益の平均 {samples.mean():.2f}(標準誤差 {samples.std() / np.sqrt(len(samples)):.2f})   "
      f"式で求めた V(0) = {V_rd[START]:.2f}")

## 8. 条件を変える

1. 本文の確認課題3:状態0 で「進む」の報酬を −1 から −4 に変えると、最適な方策はどうなるか(`R_small` を変えて、§6 を実行し直す)
2. 割引率を $\gamma = 0.5$ にすると、§6 の MDP の最適な方策は変わるか。「進む」の報酬が −2 のときはどうか
3. グリッドで `slip` を 0 と 0.5 にして、方策「右→下」の $V(0)$ がどう変わるか

In [ ]:
def best_policy(P, R, gamma):
    vals = {p: policy_value(P, R, np.array(p), gamma) for p in itertools.product(range(P.shape[1]), repeat=P.shape[0])}
    V_best = np.max(list(vals.values()), axis=0)
    return [p for p, v in vals.items() if np.allclose(v, V_best)], V_best

# 1. 最初の一歩のコストを大きくすると、状態0 では「待つ」が最適になる
R_mod = R_small.copy(); R_mod[0, 1] = -4.0
print("1. 進むの報酬 -4:", best_policy(P_small, R_mod, 0.9))

# 2. γ = 0.5:コスト -1 なら「進む」のまま、コスト -2 なら「待つ」に変わる
print("2. γ = 0.5、コスト -1:", best_policy(P_small, R_small, 0.5))
R_mod = R_small.copy(); R_mod[0, 1] = -2.0
print("   γ = 0.5、コスト -2:", best_policy(P_small, R_mod, 0.5))

# 3. 滑りやすいほど、ぬかるみに入りやすく、ゴールまでも遠くなる
for slip in [0.0, 0.2, 0.5]:
    P_s, R_s = build_grid_mdp(slip)
    print(f"3. slip = {slip}:V(0) = {policy_value(P_s, R_s, right_then_down, 0.95)[START]:.2f}")

## 9. 説明(自分の言葉で書く)

1. 収益と価値の違いは何か
   - (ここに書く)
2. ベルマン方程式 (3) の右辺の各項は、何を表しているか
   - (ここに書く)
3. 方策の価値は `solve` で求められるのに、最適な価値は求められないのはなぜか
   - (ここに書く)